# Graph End-to-End Validation Notebook
Este notebook executa os testes de uso das classes novas em sequência:
1. Carrega `.env` e autentica no Graph
2. Obtém e apresenta o conteúdo do site
3. Lista arquivos do drive e salva em DataFrame (`df_drive_items`)
4. Executa fluxo de arquivo: write, update e load
5. Executa testes de listas (views, colunas, itens, create e update)

In [2]:
from __future__ import annotations
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv

# Garante carregamento do .env a partir da raiz do repositório
repo_root = Path.cwd()
if not (repo_root / '.env').exists() and (repo_root.parent / '.env').exists():
    repo_root = repo_root.parent

env_path = repo_root / '.env'
load_dotenv(env_path)

print(f'.env carregado de: {env_path}')
print('Pronto para iniciar os testes.')

.env carregado de: c:\GitHub\MSGraphTest\.env
Pronto para iniciar os testes.


In [3]:
from msgraphtest.auth import GraphClient
from msgraphtest.drive import GraphDrive
from msgraphtest.lists import GraphList

client = GraphClient()
auth = client.authenticator
print('Cliente Graph inicializado com sucesso.')

site_attributes = {
    'sharepoint_site_id': auth.sharepoint_site_id,
    'site_graph_id': auth.site_graph_id,
    'site_name': auth.site_name,
    'site_display_name': auth.site_display_name,
    'site_web_url': auth.site_web_url,
}

print('\nAtributos do site conectado:')
for key, value in site_attributes.items():
    print(f'- {key}: {value}')

Cliente Graph inicializado com sucesso.

Atributos do site conectado:
- sharepoint_site_id: anatel365.sharepoint.com,d8124f53-4224-4434-b3ec-97f23a341b20,7fad1d86-65ba-4c88-94b9-2001d3f91ff2
- site_graph_id: anatel365.sharepoint.com,d8124f53-4224-4434-b3ec-97f23a341b20,7fad1d86-65ba-4c88-94b9-2001d3f91ff2
- site_name: sfi_app_private
- site_display_name: SFI App Dados Restritos
- site_web_url: https://anatel365.sharepoint.com/sites/sfi_app_private


In [4]:
site_contents = auth.get_site_contents()
site_data = site_contents.get('site', {})
drives_data = site_contents.get('drives', [])
lists_data = site_contents.get('lists', [])

print('Resumo do conteúdo do site:')
print(f"- Site: {site_data.get('displayName', site_data.get('name', '-'))}")
print(f"- Drives encontrados: {len(drives_data)}")
print(f"- Lists encontradas: {len(lists_data)}")

df_site_drives = pd.json_normalize(drives_data) if drives_data else pd.DataFrame()
df_site_lists = pd.json_normalize(lists_data) if lists_data else pd.DataFrame()

print('\nDrives Disponíveis:')
display(df_site_drives.head())

print('Lists Disponíveis:')
display(df_site_lists.head())

Resumo do conteúdo do site:
- Site: SFI App Dados Restritos
- Drives encontrados: 2
- Lists encontradas: 4

Drives Disponíveis:


,id,name,webUrl,driveType
0,b!U08S2CRCNESz7JfyOjQbIIYdrX-6ZYhMlLkgAdP5H_Lp...,Documentos,https://anatel365.sharepoint.com/sites/sfi_app...,documentLibrary
1,b!U08S2CRCNESz7JfyOjQbIIYdrX-6ZYhMlLkgAdP5H_I0...,GetMonitorRNI,https://anatel365.sharepoint.com/sites/sfi_app...,documentLibrary


Lists Disponíveis:


,@odata.etag,id,name,webUrl,displayName
0,"""a0db896a-d211-455c-a833-045ce1c7c40f,0""",a0db896a-d211-455c-a833-045ce1c7c40f,wte,https://anatel365.sharepoint.com/sites/sfi_app...,Extensões de Modelo da Web
1,"""4e8c9d10-db78-42b8-bdbc-3f44f1a6ce98,4""",4e8c9d10-db78-42b8-bdbc-3f44f1a6ce98,monitorRNI,https://anatel365.sharepoint.com/sites/sfi_app...,monitorRNI
2,"""7384fae9-17d5-40af-ba50-9adc5816a09b,6""",7384fae9-17d5-40af-ba50-9adc5816a09b,Documentos Compartilhados,https://anatel365.sharepoint.com/sites/sfi_app...,Documentos
3,"""a9d8be34-b989-483a-a308-e7529a99c623,10""",a9d8be34-b989-483a-a308-e7529a99c623,GetMonitorRNI,https://anatel365.sharepoint.com/sites/sfi_app...,GetMonitorRNI


In [5]:
drive = GraphDrive(client=client)
drive_items = drive.list_drive_items()

df_drive_items = pd.json_normalize(drive_items) if drive_items else pd.DataFrame()
num_files = len([item for item in drive_items if 'folder' not in item])

print(f'Total de itens no drive root: {len(drive_items)}')
print(f'Total de arquivos (sem pastas): {num_files}')
print('\nDados disponíveis:')

display(df_drive_items.head())

Total de itens no drive root: 4
Total de arquivos (sem pastas): 4

Dados disponíveis:


,@microsoft.graph.downloadUrl,createdDateTime,eTag,id,lastModifiedDateTime,name,webUrl,cTag,isAuthoritative,size,...,file.fileExtension,file.hashes.quickXorHash,file.mimeType,fileSystemInfo.createdDateTime,fileSystemInfo.lastModifiedDateTime,shared.scope,createdBy.user.email,createdBy.user.id,lastModifiedBy.user.email,lastModifiedBy.user.id
0,https://anatel365.sharepoint.com/sites/sfi_app...,2026-05-26T15:16:02Z,"""{3E8865CC-1A7D-4A50-BFAC-4253FBE37488},2""",01ZUTETVOMMWED47I2KBFL7LCCKP56G5EI,2026-05-26T15:16:05Z,notebook_test_upload_20260526_151559.txt,https://anatel365.sharepoint.com/sites/sfi_app...,"""c:{3E8865CC-1A7D-4A50-BFAC-4253FBE37488},2""",False,106,...,.txt,GvFPOUfa1FqFvTedLbv/4MGiqrg=,text/plain,2026-05-26T15:16:02Z,2026-05-26T15:16:05Z,users,NaN,NaN,NaN,NaN
1,https://anatel365.sharepoint.com/sites/sfi_app...,2026-05-26T15:41:17Z,"""{47F2A2E1-0536-4675-AD3C-DEE010F1C3ED},2""",01ZUTETVPBULZEONQFOVDK2PG64AIPDQ7N,2026-05-26T15:41:19Z,notebook_test_upload_20260526_154114.txt,https://anatel365.sharepoint.com/sites/sfi_app...,"""c:{47F2A2E1-0536-4675-AD3C-DEE010F1C3ED},2""",False,106,...,.txt,CvFNOYfa0WoFvzsNLbv/4Oliq7g=,text/plain,2026-05-26T15:41:17Z,2026-05-26T15:41:19Z,users,NaN,NaN,NaN,NaN
2,https://anatel365.sharepoint.com/sites/sfi_app...,2026-05-26T09:21:12Z,"""{DB33BC42-80D1-4948-9171-AD3E42E072D6},2""",01ZUTETVKCXQZ5XUMAJBEZC4NNHZBOA4WW,2026-05-26T09:37:37Z,sample_upload.txt,https://anatel365.sharepoint.com/sites/sfi_app...,"""c:{DB33BC42-80D1-4948-9171-AD3E42E072D6},2""",False,83,...,.txt,j0695ZwpRmmI7BCpPBzUUvCJXyc=,text/plain,2026-05-26T09:21:12Z,2026-05-26T09:37:37Z,users,NaN,NaN,NaN,NaN
3,https://anatel365.sharepoint.com/sites/sfi_app...,2026-05-26T09:00:56Z,"""{9554BB61-3CA6-4EE1-B4B3-E75F20CC91DA},1""",01ZUTETVLBXNKJLJR44FHLJM7HL4QMZEO2,2026-05-26T09:00:56Z,SharingJobLog.log,https://anatel365.sharepoint.com/sites/sfi_app...,"""c:{9554BB61-3CA6-4EE1-B4B3-E75F20CC91DA},1""",False,10498,...,.log,wC1mFC8tpe+rmBIoM/4YwwtGC4E=,application/octet-stream,2026-05-26T09:00:56Z,2026-05-26T09:00:56Z,users,lobao@anatel.gov.br,26fdc052-67a9-4c27-86ce-c541f3424d6f,lobao@anatel.gov.br,26fdc052-67a9-4c27-86ce-c541f3424d6f


## File Write, Update and Load
Este bloco cria (ou reutiliza) um arquivo de teste, atualiza o conteúdo e lê novamente para validação.

In [6]:
test_local_file = repo_root / 'downloads' / 'notebook_test_upload.txt'
test_local_file.parent.mkdir(parents=True, exist_ok=True)
test_local_file.write_text(
    f'Notebook initial content - {datetime.now(timezone.utc).isoformat()}\\n',
    encoding='utf-8',
)

upload_result = drive.upload_file(
    local_path=test_local_file,
    remote_folder='root',
    remote_name=f'notebook_test_upload_{datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")}.txt',
)
target_item_id = str(upload_result.get('id', ''))

if not target_item_id:
    raise RuntimeError('Falha ao obter item id após upload do arquivo de teste.')

print('Write concluído (upload):')
print(f"- Item ID: {target_item_id}")
print(f"- Nome: {upload_result.get('name')}")

updated_content = (
    f'Notebook updated content - {datetime.now(timezone.utc).isoformat()}\\n'
    'Linha adicional para teste de update.\\n'
    'Fim.\\n'
 )
update_result = drive.write_file_content(target_item_id, updated_content)

print('Update concluído:')
print(f"- Item ID retornado: {update_result.get('id')}")

loaded_content = drive.read_file_content(target_item_id)
print('Load concluído (conteúdo atual lido):')
print(loaded_content)

download_path = repo_root / 'downloads' / f'downloaded_{target_item_id}.txt'
saved_path = drive.download_file(target_item_id, download_path)
print(f'Arquivo carregado para disco local em: {saved_path}')

Write concluído (upload):
- Item ID: 01ZUTETVPJMGCAHI7O7ZH3B254SRG4H6OV
- Nome: notebook_test_upload_20260526_164041.txt
Update concluído:
- Item ID retornado: 01ZUTETVPJMGCAHI7O7ZH3B254SRG4H6OV
Load concluído (conteúdo atual lido):
Notebook updated content - 2026-05-26T16:40:43.004725+00:00\nLinha adicional para teste de update.\nFim.\n
Arquivo carregado para disco local em: C:\GitHub\MSGraphTest\downloads\downloaded_01ZUTETVPJMGCAHI7O7ZH3B254SRG4H6OV.txt


## List Tests
Este bloco executa os testes de lista: views, colunas, itens, create e update.

In [10]:
list_client = GraphList(client=client)

list_views = list_client.get_list_views()
list_columns = list_client.get_list_columns()
list_items = list_client.get_list_items(
    include_title=True,
    fields_only=True,
    include_item_id=True,
)

print(f'Views encontradas: {len(list_views)}')
print(f'Colunas consultadas (filtradas): {len(list_columns)}')
print(f'Itens encontrados: {len(list_items)}')


column_rename_map = {
    col["name"]: col.get("displayName", col["name"])
    for col in list_columns
    if col.get("name")
}
column_rename_map["id"] = "Item ID"
df_list_items = pd.DataFrame(list_items).rename(columns=column_rename_map) if list_items else pd.DataFrame()

print('DataFrame `df_list_items` criado (sem metadados internos, mantendo o Item ID).')
display(df_list_items.head())

test_title = f"Notebook item {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}"
created_item = list_client.create_list_item({'Title': test_title})
created_item_id = str(created_item.get('id', ''))

if not created_item_id:
    raise RuntimeError('Falha ao obter o ID do item criado na lista.')

print('Create de item concluído:')
print(f'- Item ID: {created_item_id}')
print(f"- Título inicial: {test_title}")

updated_title = f"{test_title} (updated)"
update_result = list_client.update_list_item(created_item_id, {'Title': updated_title})

print('Update de item concluído:')
print(update_result)

refreshed_items = list_client.get_list_items(
    include_title=True,
    fields_only=True,
    include_item_id=True,
)
df_list_items_after_update = pd.DataFrame(refreshed_items).rename(columns=column_rename_map) if refreshed_items else pd.DataFrame()
print(f'Total de itens após create/update: {len(refreshed_items)}')
display(df_list_items_after_update.head())

Views encontradas: 0
Colunas consultadas (filtradas): 41
Itens encontrados: 7
DataFrame `df_list_items` criado (sem metadados internos, mantendo o Item ID).


,@odata.etag,Title,Item ID
0,"""2b08bda2-22e2-435f-bae6-1177f3ae8b25,5""",Updated by msgraphtest,1
1,"""57efbf87-42e2-46fb-ba5e-31e1c3036765,1""",Test item created by msgraphtest,2
2,"""db1e0383-aabb-457f-ad7a-80d3fe8c6904,2""",Notebook item 2026-05-26 15:44:45 (updated),3
3,"""7dca8101-328a-4bcc-a348-ae0f2301c151,2""",Notebook item 2026-05-26 16:06:24 (updated),4
4,"""9a68aa68-3922-4c41-9c4a-6cc19100be3e,2""",Notebook item 2026-05-26 16:40:53 (updated),5


Create de item concluído:
- Item ID: 8
- Título inicial: Notebook item 2026-05-26 17:00:33
Update de item concluído:
{'@odata.context': "https://graph.microsoft.com/v1.0/$metadata#sites('anatel365.sharepoint.com%2Cd8124f53-4224-4434-b3ec-97f23a341b20%2C7fad1d86-65ba-4c88-94b9-2001d3f91ff2')/lists('4e8c9d10-db78-42b8-bdbc-3f44f1a6ce98')/items('8')/fields/$entity", '@odata.etag': '"e5255ca1-f554-4909-9124-6c576dfcd998,2"', 'Title': 'Notebook item 2026-05-26 17:00:33 (updated)', 'LinkTitle': 'Notebook item 2026-05-26 17:00:33 (updated)', 'id': '8', 'ContentType': 'Item', 'Modified': '2026-05-26T17:00:37Z', 'Created': '2026-05-26T17:00:36Z', 'AuthorLookupId': '1073741822', 'EditorLookupId': '1073741822', '_UIVersionString': '2.0', 'Attachments': False, 'Edit': '', 'LinkTitleNoMenu': 'Notebook item 2026-05-26 17:00:33 (updated)', 'ItemChildCount': '0', 'FolderChildCount': '0', '_ComplianceFlags': '', '_ComplianceTag': '', '_ComplianceTagWrittenTime': '', '_ComplianceTagUserId': '', 'AppAuth

,@odata.etag,Title,Item ID
0,"""2b08bda2-22e2-435f-bae6-1177f3ae8b25,5""",Updated by msgraphtest,1
1,"""57efbf87-42e2-46fb-ba5e-31e1c3036765,1""",Test item created by msgraphtest,2
2,"""db1e0383-aabb-457f-ad7a-80d3fe8c6904,2""",Notebook item 2026-05-26 15:44:45 (updated),3
3,"""7dca8101-328a-4bcc-a348-ae0f2301c151,2""",Notebook item 2026-05-26 16:06:24 (updated),4
4,"""9a68aa68-3922-4c41-9c4a-6cc19100be3e,2""",Notebook item 2026-05-26 16:40:53 (updated),5
